# Notes Generator

In [44]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

import requests
import json

In [24]:
model = ChatOllama(model="qwen3-coder:30b", temperature=0.9)

Tavily Integration

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Add TAVILY_API_KEY to your .env file: TAVILY_API_KEY=your_key_here
API_KEY = os.environ.get("TAVILY_API_KEY", "")
BASE_URL = "https://api.tavily.com/search"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

In [ ]:
def perform_web_search(query: str) -> dict:
    payload = {
        "query": f"Tell me about {query}",
        "topic": "general",
        "search_depth": "advanced",
        "max_results": 10,
        "time_range": None,
        "days": 30,
        "include_answer": "basic",
        "include_raw_content": True,
        "include_images": False,
        "include_image_descriptions": False,
        "include_domains": ["https://www.wired.com/"],
    }
    r = requests.post(BASE_URL, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()
    

In [37]:
def format_tavily(t: dict) -> str:
    # 1) Use the built-in synthesized answer if present
    parts = []
    if t.get("answer"):
        parts.append(f"Provider summary answer:\n{t['answer']}\n")

    # 2) Add sources (prefer raw_content when available)
    results = t.get("results", [])
    for i, r in enumerate(results, 1):
        title = r.get("title", "").strip()
        url = r.get("url", "").strip()
        text = (r.get("raw_content") or r.get("content") or "").strip()

        # Keep it bounded so you don't blow context
        if len(text) > 2000:
            text = text[:2000] + "…"

        parts.append(f"[{i}] {title}\n{url}\n{text}\n")

    return "\n".join(parts) if parts else "No web results returned."

In [39]:
web_context = RunnableLambda(perform_web_search) | RunnableLambda(format_tavily)
web_context

RunnableLambda(perform_web_search)
| RunnableLambda(format_tavily)

In [27]:
topic = str(input("Enter a topic to to generate notes and quiz on: "))

In [ ]:
topicPrompt = PromptTemplate(
    input_variables=["topic", "data"],
    template=(
        "Do a comprehensive brainstorming on {topic} "
        "using these results from the web:\n\n{data}"
    ),
)

In [ ]:
topicInfoChain = web_context | RunnableLambda(lambda web_data: topicPrompt.format(topic=topic, data=web_data)) | model | StrOutputParser()

In [45]:
generateNotes = PromptTemplate(
    template="Generate structured notes using the following web search results: {content}",
    input_variables=["content"],
)

In [46]:
notesToQuizPrompt = PromptTemplate(
    template="Generate a quiz with 5 questions based on the following web search results: {content}",
    input_variables=["content"],
)

In [47]:
materialsChain = RunnableParallel(
    {
        "notes": generateNotes | model | StrOutputParser(),
        "quiz": notesToQuizPrompt | model | StrOutputParser(),
    }
)

In [48]:
fullChain = topicInfoChain | materialsChain

In [49]:
fullChain.invoke(topic)

{'notes': "# Structured Notes: Quantum Computing Brainstorming\n\n## I. Introduction\n- **Definition**: Revolutionary paradigm shift in computational processing using quantum mechanics\n- **Purpose**: Solving problems intractable for classical computers\n- **Scope**: Explores fundamental concepts, applications, challenges, and future prospects\n\n## II. Core Quantum Mechanics Principles\n\n### A. Superposition\n- **Concept**: Qubits exist in multiple states simultaneously (0 and 1 at same time)\n- **Implication**: Enables parallel processing on exponential scale\n- **Mathematical representation**: |ψ⟩ = α|0⟩ + β|1⟩ where α² + β² = 1\n\n### B. Entanglement\n- **Concept**: Qubits become correlated; measuring one affects the other instantly\n- **Application**: Quantum teleportation and secure communication\n- **Significance**: Creates non-local correlations impossible for classical systems\n\n### C. Quantum Interference\n- **Concept**: Quantum states amplify correct answers and cancel wro